# Learning from doing vs watching: the geometry of two evidence streams

> **Chen, Radkani, Valluru, Yoo & Jazayeri (2026), *[Evidence accumulation from experience and observation in the cingulate cortex](https://doi.org/10.1038/s41586-025-09885-0)*, Nature 650, 681–694**.

## Background: what the brain shows, and why we need a model

**The task.** Two monkeys share a betting task. A hidden "good" target pays out and
occasionally switches without warning. The **actor** plays and sees its own wins and
losses; the **observer** only watches, then takes its turn. To play well, both must
*integrate outcomes across trials* — every loss is evidence that the good target has
moved — whether the outcome happened to them or to the other monkey.

In the anterior cingulate cortex (ACC), population activity
contains a *switch-evidence* axis: activity along it grows with every consecutive loss,
for own and for observed outcomes alike — but it grows faster for experienced ones,
matching the behavioural asymmetry (choices track experienced outcomes more strongly than
observed ones). Geometrically, the direction that encodes **own** outcomes and the
direction that encodes the **other's** outcomes are close to orthogonal in both monkeys
(90.8° in M1, 76.0° in M2), and the observer direction sits *farther* from the
switch-evidence axis than the actor direction (M1: 79.1° vs 26.3°; M2: 54.4° vs 32.0°).

**Why train RNNs?** Recordings reveal this geometry but not the wiring that creates it.
Two candidate wirings differ only in how outcomes enter the network:

* **H1 — shared input.** One identity-agnostic outcome channel plus a separate
  "whose outcome" flag. The same wires carry both streams, which pushes the two
  outcome signals toward the **same** coding direction.
* **H2 — separate inputs.** One outcome channel per source, only one active at a time.
  Nothing ties the two encodings together, so they are free to land in **different**
  directions.

Train identical networks under each wiring, measure the same angles that were measured in
ACC, and ask which wiring's geometry looks like the brain's.

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

import neuralrnn  # registers all model families (import side effect)
from neuralrnn import (AutoModel, SupervisedObjective, Trainer, TrainingArguments)

NOTEBOOK_DIR = Path.cwd()
MODEL_DIR = NOTEBOOK_DIR / "models" / "evidence_asymmetry"   # trained checkpoints
FIG_DIR = NOTEBOOK_DIR / "figs" / "evidence_asymmetry"       # figures go here
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"   # 500 sequential steps of tiny matmuls: CPU beats GPU
SEEDS = range(6)  # six networks per wiring -> the angle distributions

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed()

## The task: evidence integration, distilled

The MSI (multi-source integration) task keeps only what matters for the question. One
episode is 500 timesteps (~35 trials). On each trial the network receives **one outcome**
— a scalar drawn from $N(+0.5,\ 0.1^2)$ if the trial was rewarded or $N(-0.5,\ 0.1^2)$ if
not — tagged as belonging to the actor or the observer. A go cue opens a short **report
window** in which the network must output the accumulated evidence

$$Y = (\text{rewards}) \;-\; g\cdot(\text{unrewarded}), \qquad
g = \begin{cases} 1 & \text{actor trials} \\ 0.5 & \text{observer trials}\end{cases}$$

which resets to zero after every rewarded trial. Outside the report window the target is
undefined and masked out of the loss. The built-in gain $g$ is what makes observed
outcomes count for less.

The two wirings differ **only** in how the three input channels are used:

| channel | H1 "paral" — shared input | H2 "ortho" — separate inputs |
|---|---|---|
| 1 | outcome value (either source) | actor outcome |
| 2 | identity flag (+1 actor / −1 observer) | observer outcome |
| 3 | go cue | go cue |

In [ ]:
from __future__ import annotations

import numpy as np
import torch

# paper task config (configs/msi_rnn.py)
PAPER_TASK_KWARGS = dict(
    timesteps=500,
    integration_factor=0.5,    # observer gain (actor gain = 1)
    mean_choices=[-0.5, 0.5],  # unrewarded / rewarded Gaussian modes
    value_std=0.1,
    iti=5,
)


class MSI:
    """Multi-Source-Integration task generator (one call = one episode)."""

    def __init__(
        self,
        timesteps: int,
        model_type: str,
        integration_factor: float,
        mean_choices,
        value_std: float,
        test: bool = False,
        iti: int = 5,
        rng=None,
    ):
        assert model_type in ("paral", "ortho")
        self._timesteps = int(timesteps)
        self._model_type = model_type
        self._integration_factor = float(integration_factor)
        self._mean_choices = list(mean_choices)
        self._value_std = float(value_std)
        self._test = bool(test)
        self._risetime = int(iti)
        self._rng = rng  # unused; all randomness uses the global np.random state

    # ------------------------------------------------------------------ #
    def generate_identity(self, trials):
        """Per-timestep identity sequence: +1 actor onset, -1 observer, else 0."""
        tstarts = trials["trialstarts"]
        ntrials = len(tstarts)
        identities = np.ones(ntrials)
        if self._test:
            identities[ntrials // 2:] = -1
        else:
            identities = np.random.choice([1, -1], size=ntrials)
        sequence = np.zeros(self._timesteps)
        for ts, ident in zip(tstarts, identities):
            sequence[ts] = ident
        return sequence

    def generate_means(self, trials):
        """Per-timestep outcome-mode sequence (the Gaussian mean per onset)."""
        tstarts = trials["trialstarts"]
        ntrials = len(tstarts)
        means = np.random.choice(self._mean_choices, size=ntrials)
        if self._test:  # fixed layout: NR, R | NR, R with actor first half
            means[:ntrials // 4] = self._mean_choices[0]
            means[ntrials // 4:ntrials // 2] = self._mean_choices[1]
            means[ntrials // 2:3 * ntrials // 4] = self._mean_choices[0]
            means[3 * ntrials // 4:] = self._mean_choices[1]
        sequence = np.zeros(self._timesteps)
        for ts, m in zip(tstarts, means):
            sequence[ts] = m
        return sequence

    def generate_trials(self):
        """Random trial layout: starts + go-cue (report ramp) starts."""
        trials = {"trialstarts": [], "rampstarts": []}
        start = 0
        ramp = self._risetime
        max_len = ramp * 4   # 20 steps at iti=5
        min_len = ramp * 2   # 10 steps
        while start + max_len < self._timesteps:
            t_trial = np.random.randint(min_len, max_len)  # 10..19
            total_iti = t_trial - 2 * ramp                 # split into delay/iti
            delay = np.random.randint(0, total_iti) if total_iti > 0 else 0
            trials["trialstarts"].append(start)
            trials["rampstarts"].append(start + delay)
            start += t_trial
        return trials

    # ------------------------------------------------------------------ #
    def __call__(self):
        trials = self.generate_trials()
        identity = self.generate_identity(trials)      # (T,) at onsets
        means = self.generate_means(trials)            # (T,)
        values = np.random.normal(means, self._value_std)  # (T,)
        values[identity == 0] = 0                      # only onsets carry values

        # ---- input channels depending on the architectural hypothesis ----
        if self._model_type == "ortho":                # H2: separate inputs
            stim_self = [v if ident == 1 else 0 for v, ident in zip(values, identity)]
            stim_other = [v if ident == -1 else 0 for v, ident in zip(values, identity)]
            inputs = np.column_stack([stim_self, stim_other])
        else:                                          # H1 ("paral")
            inputs = np.column_stack([values, identity])

        # ---- evidence accumulator (the desired output) -------------------
        integration_factors = np.zeros(self._timesteps)
        integration_factors[identity == 1] = 1.0
        integration_factors[identity == -1] = self._integration_factor
        scaled_input = values * integration_factors

        accumulator = 0.0
        output = []
        for value, scaled in zip(values, scaled_input):
            if value > 0:      # rewarded trial: reset (paper constraint 1)
                accumulator = 0.0
            else:              # unrewarded: integrate with source gain
                accumulator += scaled
            output.append(accumulator)
        output = np.array(output)

        # ---- go cue (report ramp) and nan-masking of the label -----------
        input_report = np.zeros(self._timesteps)
        for ramp_start in trials["rampstarts"]:
            ramp_end = ramp_start + self._risetime
            report_off = ramp_end + self._risetime
            input_report[ramp_start:ramp_end] = np.linspace(0, 1, self._risetime)
            input_report[ramp_end:report_off] = 1.0
        output = np.array(
            [o if report == 1 else np.nan for o, report in zip(output, input_report)]
        )

        inputs = np.column_stack([inputs, input_report])          # (T, 3)
        return {
            "inputs": inputs[:, :, None].astype(np.float32),      # (T, 3, 1)
            "labels": output[:, None].astype(np.float32),         # (T, 1)
            "identity": identity,                                 # (T,)
            "trialstarts": np.array(trials["trialstarts"]),
            "rampstarts": np.array(trials["rampstarts"]),
        }

    @property
    def data_keys(self):
        return ("inputs", "labels")


# ---------------------------------------------------------------------- #
# NeuralRNN data-layer adapter
# ---------------------------------------------------------------------- #
def make_dataset(model_type: str, n_episodes: int = 1024, batch_size: int = 16,
                 seed: int = 0):
    """Build a ``CognitiveTaskDataset`` (aligned mode) of MSI episodes.

    One dataset "trial" is one full 500-step episode (~30 trials of the
    behavioural task). NaN labels (outside the report windows) are replaced by
    0 and carried by the mask — the framework's loss contract.

    We pre-generate a pool of ``n_episodes`` episodes and sample batches
    with replacement (aligned mode), the framework's standard data contract.

    Returns:
        CognitiveTaskDataset with sample_batch() -> {"inputs" (B,T,3),
        "targets" (B,T,1), "mask" (B,T,1)}.
    """
    from neuralrnn import CognitiveTaskDataset

    np.random.seed(seed)
    task = MSI(model_type=model_type, **PAPER_TASK_KWARGS)
    inputs, targets, masks, conditions = [], [], [], []
    for ep_idx in range(n_episodes):
        ep = task()
        x = ep["inputs"][..., 0]                    # (T, 3)
        y = ep["labels"]                            # (T, 1), NaN off-report
        m = ~np.isnan(y)
        inputs.append(torch.from_numpy(x))
        targets.append(torch.from_numpy(np.where(m, y, 0.0)))
        masks.append(torch.from_numpy(m.astype(np.float32)))
        conditions.append({
            "n_steps": x.shape[0],
            "epochs": {},
            "is_catch": False,
            "episode": ep_idx,
            "trialstarts": ep["trialstarts"].tolist(),
            "rampstarts": ep["rampstarts"].tolist(),
        })
    return CognitiveTaskDataset(
        inputs=torch.stack(inputs),
        targets=torch.stack(targets),
        mask=torch.stack(masks),
        conditions=conditions,
        task_name=f"msi_{model_type}",
        batch_size=batch_size,
        mode="aligned",
    )

# one sample episode per wiring — each input channel on its own row
CH_LABELS = {
    "paral": ["outcome value", "identity", "report cue"],
    "ortho": ["actor outcome", "observer outcome", "report cue"],
}
CH_COLORS = {
    "paral": ["k", "purple", "tab:green"],
    "ortho": ["tab:blue", "tab:red", "tab:green"],
}
CH_YLIMS = {
    "paral": [(-1.2, 1.2), (-1.6, 1.6), (-0.15, 1.25)],
    "ortho": [(-1.2, 1.2), (-1.2, 1.2), (-0.15, 1.25)],
}

fig, axs = plt.subplots(4, 2, figsize=(10, 6.8), sharex="col",
                        gridspec_kw=dict(height_ratios=[1, 1, 1, 1.6]))
for col, (mtype, title) in enumerate([
        ("paral", "H1: shared outcome + identity flag (+1 actor / −1 observer)"),
        ("ortho", "H2: one outcome channel per source")]):
    np.random.seed(col)
    d = MSI(model_type=mtype, **PAPER_TASK_KWARGS)()
    x = np.squeeze(d["inputs"])          # (T, 3) input channels
    y = np.squeeze(d["labels"])          # (T,) target evidence, NaN outside report
    t = np.arange(len(y))
    go = x[:, 2] == 1

    ch_axs = [axs[r, col] for r in range(3)]   # one row per input channel
    ax_out = axs[3, col]
    # shade the report windows (contiguous go-cue plateaus) on every row
    edges = np.flatnonzero(np.diff(np.concatenate(([0], go.view(np.int8), [0]))))
    for a, b in edges.reshape(-1, 2):
        for ax in (*ch_axs, ax_out):
            ax.axvspan(a, b, color="grey", alpha=0.10, lw=0)
    # each channel on its own row so the traces never overlap
    for ax, ch in zip(ch_axs, range(3)):
        ax.plot(t, x[:, ch], color=CH_COLORS[mtype][ch], lw=0.9)
        ax.set_ylabel(CH_LABELS[mtype][ch], fontsize=8)
        ax.set_ylim(*CH_YLIMS[mtype][ch])
        ax.tick_params(labelsize=8)
    # outcome onsets, green = rewarded, red = unrewarded
    if mtype == "paral":
        val, ident = x[:, 0], x[:, 1]
        onsets = np.flatnonzero((val != 0) & (ident != 0))
        for s in onsets:
            rew = val[s] > 0
            ch_axs[0].plot(s, val[s], "^" if rew else "v", ms=5,
                           color="green" if rew else "red")
    else:
        onsets = np.flatnonzero((x[:, 0] != 0) | (x[:, 1] != 0))
        for s in onsets:
            ch = 0 if x[s, 0] != 0 else 1
            v = x[s, ch]
            ch_axs[ch].plot(s, v, "^" if v > 0 else "v", ms=5,
                            color="green" if v > 0 else "red")
    ch_axs[0].plot([], [], "^", color="green", label="rewarded trial")
    ch_axs[0].plot([], [], "v", color="red", label="unrewarded trial")
    ch_axs[0].legend(fontsize=7, ncol=2, loc="upper left", framealpha=0.95)
    ch_axs[0].set_title(title, fontsize=10)

    ax_out.plot(t, y, color="k", lw=1.2)
    ax_out.set_ylabel("target Y")
    ax_out.set_xlabel("time (steps)")
    ax_out.tick_params(labelsize=8)
    for ax in (*ch_axs, ax_out):
        ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("One episode per wiring — grey bands: report windows (only there is Y defined)",
             fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_task.png", dpi=150)
plt.show()

## The network: one equation, two wirings

The paper's model (Methods, eq. 11–12) is a continuous-time firing-rate RNN of $N=200$
tanh units with time constant $\tau = 5$ steps,

$$\tau \frac{dx}{dt} = -x + W_e I(t) + W_i \tanh(x) + b, \qquad Y = W_o x .$$

Euler-discretized with step 1 ($\alpha = 1/\tau = 0.2$) this is exactly the framework's
`CTRNNModel` with `nonlinearity_mode="rate"`:

$$x' = (1-\alpha)\,x + \alpha\,(W_i \tanh(x) + W_e I + b).$$

Following the paper, the input weights $W_e$ and the readout $W_o$ are drawn randomly and
**frozen**; only the recurrent matrix $W_i$ is trained (Adam, lr $10^{-4}$, masked MSE on
the report windows, 20k iterations — the paper used 100k; the qualitative geometry
converges much earlier). The two wirings share everything except the input channels.

To see a *distribution* of angles rather than a single point, we train **six networks per
wiring** (seeds 0–5). Training is **load-first**: once a checkpoint exists in
`models/evidence_asymmetry/`, rerunning this notebook takes seconds.

In [ ]:
from __future__ import annotations

import torch

from neuralrnn import AutoConfig, AutoModel

HIDDEN_SIZE = 200
ACTIVITY_DECAY = 0.2   # alpha = dt/tau = 1/5 (paper Methods: tau = 5)

# variant name -> (model_type, learn_output, theta_reg_weight)
VARIANTS = {
    "h1fix":   ("paral", False, 0.0),
    "h2fix":   ("ortho", False, 0.0),
    "h1learn": ("paral", True, 0.0),
    "h2learn": ("ortho", True, 0.0),
    "h1wide":  ("paral", False, 1.0),
    "h2wide":  ("ortho", False, 1.0),
}


def make_config(model_type: str, learn_output: bool = False):
    """CTRNNConfig implementing the paper's architecture + constraints."""
    assert model_type in ("paral", "ortho")
    return AutoConfig.for_model(
        "ctrnn",
        input_dim=3,             # [self/ value, other/identity, go cue]
        latent_dim=HIDDEN_SIZE,
        output_dim=1,            # scalar accumulated evidence Y
        activation="tanh",
        alpha=ACTIVITY_DECAY,
        nonlinearity_mode="rate",   # x' = (1-a)x + a(W tanh(x) + B I + b)
        freeze_input=True,          # input weights random & fixed (all variants)
        freeze_output=not learn_output,
    )


def make_model(model_type: str, learn_output: bool = False, seed: int = 0):
    """Randomly initialized framework CTRNN for one MSI variant."""
    torch.manual_seed(seed)
    return AutoModel.from_config(make_config(model_type, learn_output))


def train_or_load(variant, seed):
    """Train one network with the framework Trainer, or load it if already done."""
    model_type, learn_output, _ = VARIANTS[variant]
    out_dir = MODEL_DIR / f"{variant}_s{seed}"
    if (out_dir / "run_meta.json").exists() and (out_dir / "config.json").exists():
        return AutoModel.from_pretrained(str(out_dir))
    out_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    dataset = make_dataset(model_type, n_episodes=1024, batch_size=16, seed=seed)
    model = make_model(model_type, learn_output=learn_output, seed=seed)
    args = TrainingArguments(learning_rate=1e-4, max_steps=20_000,
                             grad_clip_norm=None, log_every=500,
                             output_dir=str(out_dir), device=DEVICE, seed=seed)
    Trainer(model, dataset, SupervisedObjective(task_type="regression"), args).train()
    model.save_pretrained(str(out_dir))
    (out_dir / "run_meta.json").write_text(
        json.dumps({"variant": variant, "seed": seed}))
    return model


models = {}   # (variant, seed) -> trained CTRNNModel
for variant in ("h1fix", "h2fix"):
    for seed in SEEDS:
        models[variant, seed] = train_or_load(variant, seed)
        print(f"{variant} seed {seed}: ready")

## The angle analysis: coding directions and the angle between them

The question is geometric: along which direction of the 200-dim population space does the
network encode "my outcome", and along which "their outcome"?

1. **Coding direction of a signal.** For each unit $i$, take its activation at all trial
   onsets of one source (say the actor's) and linearly regress it onto the outcome sign
   ($+1$ rewarded / $-1$ unrewarded). The coefficient $\beta_i$ says how strongly that
   unit carries the signal. Collected over all units, $\beta \in \mathbb{R}^{200}$ is the
   signal's *coding direction* — $O^{Act}$ for actor outcomes, $O^{Obs}$ for observer
   outcomes.
2. **Angle between directions.** $\theta(O^{Act}, O^{Obs}) = \arccos\frac{\langle
   O^{Act}, O^{Obs}\rangle}{|O^{Act}|\,|O^{Obs}|}$. Near $0°$: the same neurons carry
   both signals (shared code). Near $90°$: independent subspaces.
3. **Distance to the belief.** The same recipe against the network's own output at
   report time gives the evidence direction $SE$ ("switch evidence"), and the angles
   $\theta(O^{Act}, SE)$, $\theta(O^{Obs}, SE)$ measure how strongly each stream can move
   the belief. If the observer direction points farther away from $SE$, observed outcomes
   move the belief less — a geometric explanation of the behavioural asymmetry.

Estimates from a single episode (~35 trials) are noisy, so per network we average over 5
drawn episodes, and we look at the distribution across the six networks of each wiring.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import torch
from sklearn.linear_model import LinearRegression



def compute_angle(v1, v2) -> float:
    """Angle (degrees) between two vectors."""
    return np.degrees(np.arccos(
        np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))))


def get_history(trial_data: dict, model_type: str) -> np.ndarray:
    """Trial history label per timestep: -1 = 1R, 0 = 1NR, 1,2,... = nNR."""
    inputs = np.squeeze(trial_data["inputs"])        # (T, 3)
    if model_type == "paral":
        values = inputs[:, 0]
        identity = inputs[:, 1]
    else:
        value_self, value_other = inputs[:, 0], inputs[:, 1]
        values = value_self + value_other
        identity = np.zeros_like(values)
        identity[value_self != 0] = 1
        identity[value_other != 0] = -1
    history = np.full_like(values, np.nan)
    history[values > 0] = -1     # rewarded
    history[values < 0] = 0      # first unrewarded
    onsets = np.where(~np.isnan(history))[0]
    for prev, ts in zip(onsets[:-1], onsets[1:]):
        if history[prev] > -1 and history[ts] > -1:   # consecutive NR trials
            history[ts] = history[prev] + 1
    for i in range(1, len(history)):                 # hold value between onsets
        if np.isnan(history[i]):
            history[i] = history[i - 1]
    return history.squeeze()


@torch.no_grad()
def run_episode(model, trial_data: dict):
    """Run one episode through a framework model -> (states (T,N), outputs (T,))."""
    model.eval()
    inputs = torch.from_numpy(trial_data["inputs"][..., 0]).unsqueeze(0)  # (1,T,3)
    out = model(inputs)
    states = out.states[0].numpy()                    # (T, N)
    outputs = out.outputs[0, :, 0].numpy()            # (T,)
    return states, outputs


def compute_thetas(model, model_type: str,
                   n_time: int = 5, random_seed: int = 0) -> dict:
    """Angle analysis of one trained network.

    For each offset 0..n_time-1 after the stimulus onsets:
      * ``coef_outdir`` : per-unit regression of activity -> network output at
        the report onsets (the "output/evidence" direction);
      * ``coef_self`` / ``coef_other`` : per-unit regression of activity at
        (onset + offset) -> outcome sign (+1 R / -1 NR) for actor / observer
        trials (the outcome coding directions);
      * theta_self_other : angle between actor and observer directions
        (Fig. 4h/l);  theta_*_out : angle with the output direction
        (Fig. 4r/s).

    The stimulus is drawn from the *global* numpy RNG (reference quirk), so we
    seed it with ``random_seed`` for reproducibility.
    """
    np.random.seed(random_seed)
    n_units = model.config.latent_dim
    task = MSI(model_type=model_type, **PAPER_TASK_KWARGS)

    trial_data = task()
    ramp_onsets = trial_data["rampstarts"]
    history = get_history(trial_data, model_type)
    identity = trial_data["identity"].copy()
    idx_input = np.where(identity != 0)[0]          # all trial onsets
    for i in range(1, len(identity)):               # forward-fill identity
        if identity[i] == 0:
            identity[i] = identity[i - 1]

    states, outputs_array = run_episode(model, trial_data)
    activations = states.T                          # (N, T)
    test_inputs = trial_data["inputs"].squeeze()    # (T, 3)
    test_outputs = trial_data["labels"].squeeze()   # (T,)

    # first flat-1 timestep of every report window
    idx_report = np.where(test_inputs[:, 2] == 1)[0]
    idx_report_first = [idx_report[0]]
    prev = idx_report[0]
    for idx in idx_report:
        if idx - prev > 1:
            idx_report_first.append(idx)
        prev = idx
    idx_self_report = np.intersect1d(np.where(identity == 1)[0], idx_report)
    idx_other_report = np.intersect1d(np.where(identity == -1)[0], idx_report)

    target = test_outputs[idx_report]
    output_reporting = outputs_array[idx_report]
    history_reporting = history[idx_report]
    target_self = test_outputs[idx_self_report]
    output_self = outputs_array[idx_self_report]
    history_self = history[idx_self_report]
    target_other = test_outputs[idx_other_report]
    output_other = outputs_array[idx_other_report]
    history_other = history[idx_other_report]

    self_trials = np.intersect1d(np.where(identity == 1)[0], idx_input)
    other_trials = np.intersect1d(np.where(identity == -1)[0], idx_input)

    theta_self_other = np.zeros(n_time)
    theta_self_out = np.zeros(n_time)
    theta_other_out = np.zeros(n_time)
    theta_self_out_ramp = np.zeros(n_time)
    theta_other_out_ramp = np.zeros(n_time)
    betas_self = np.zeros([n_units, n_time])
    betas_other = np.zeros([n_units, n_time])
    betas_outdir = np.zeros([n_units, n_time])

    for i_off in range(n_time):
        # output (evidence) direction: activity at onsets+off -> report output
        x = activations[:, idx_input + i_off]
        y = outputs_array[idx_report_first]
        for i_unit in range(n_units):
            reg = LinearRegression(fit_intercept=True)
            reg.fit(x[i_unit, :].reshape(-1, 1), y)
            betas_outdir[i_unit, i_off] = reg.coef_[0]

        # actor direction
        x = activations[:, self_trials + i_off]
        y = test_inputs[self_trials, 0].copy()
        y[y > 0], y[y < 0] = 1, -1
        for i_unit in range(n_units):
            reg = LinearRegression(fit_intercept=True)
            reg.fit(x[i_unit, :].reshape(-1, 1), y)
            betas_self[i_unit, i_off] = reg.coef_[0]

        # observer direction
        x = activations[:, other_trials + i_off]
        y = test_inputs[other_trials, 1 if model_type == "ortho" else 0].copy()
        y[y > 0], y[y < 0] = 1, -1
        for i_unit in range(n_units):
            reg = LinearRegression(fit_intercept=True)
            reg.fit(x[i_unit, :].reshape(-1, 1), y)
            betas_other[i_unit, i_off] = reg.coef_[0]

        # ramp-aligned variants (offset relative to the go-cue onsets)
        self_ramp = np.intersect1d(np.where(identity == 1)[0], ramp_onsets) + i_off
        other_ramp = np.intersect1d(np.where(identity == -1)[0], ramp_onsets) + i_off
        y = test_inputs[self_trials, 0].copy()
        y[y > 0], y[y < 0] = 1, -1
        x = activations[:, self_ramp]
        bs = np.zeros(n_units)
        for i_unit in range(n_units):
            reg = LinearRegression(fit_intercept=True)
            reg.fit(x[i_unit, :].reshape(-1, 1), y)
            bs[i_unit] = reg.coef_[0]
        y = test_inputs[other_trials, 1 if model_type == "ortho" else 0].copy()
        y[y > 0], y[y < 0] = 1, -1
        x = activations[:, other_ramp]
        bo = np.zeros(n_units)
        for i_unit in range(n_units):
            reg = LinearRegression(fit_intercept=True)
            reg.fit(x[i_unit, :].reshape(-1, 1), y)
            bo[i_unit] = reg.coef_[0]
        theta_self_out_ramp[i_off] = compute_angle(bs, betas_outdir[:, i_off])
        theta_other_out_ramp[i_off] = compute_angle(bo, betas_outdir[:, i_off])
        theta_other_out[i_off] = compute_angle(
            betas_other[:, i_off], betas_outdir[:, i_off])
        theta_self_out[i_off] = compute_angle(
            betas_self[:, i_off], betas_outdir[:, i_off])
        theta_self_other[i_off] = compute_angle(
            betas_self[:, i_off], betas_other[:, i_off])

    return {
        "target": target.tolist(),
        "output": output_reporting.tolist(),
        "history": history_reporting.tolist(),
        "target_self": target_self.tolist(),
        "output_self": output_self.tolist(),
        "history_self": history_self.tolist(),
        "target_other": target_other.tolist(),
        "output_other": output_other.tolist(),
        "history_other": history_other.tolist(),
        "theta_self_other": theta_self_other.tolist(),
        "theta_self_out": theta_self_out.tolist(),
        "theta_other_out": theta_other_out.tolist(),
        "theta_self_out_ramp": theta_self_out_ramp.tolist(),
        "theta_other_out_ramp": theta_other_out_ramp.tolist(),
        "coef_self": betas_self.tolist(),
        "coef_other": betas_other.tolist(),
        "coef_outdir": betas_outdir.tolist(),
    }


def compute_thetas_avg(model, model_type: str, n_episodes: int = 5,
                       random_seed: int = 0, **kwargs) -> dict:
    """Average the angle estimates of :func:`compute_thetas` over
    ``n_episodes`` independently drawn stimulus episodes (single-episode
    estimates are noisy with ~35 trials, so we average).
    """
    acc = None
    first = None
    keys = ("theta_self_other", "theta_self_out", "theta_other_out",
            "theta_self_out_ramp", "theta_other_out_ramp")
    for ep in range(n_episodes):
        res = compute_thetas(model, model_type, random_seed=random_seed + ep,
                             **kwargs)
        if first is None:
            first = res
        if acc is None:
            acc = {k: np.array(res[k], float) for k in keys}
        else:
            for k in keys:
                acc[k] += np.array(res[k], float)
    out = dict(first)
    for k in acc:
        out[k] = (acc[k] / n_episodes).tolist()
    return out

ANGLE_KEYS = {"theta_self_other": r"$\theta(O^{Act}, O^{Obs})$",
              "theta_self_out":   r"$\theta(O^{Act}, SE)$",
              "theta_other_out":  r"$\theta(O^{Obs}, SE)$"}

# per-network angles (offset 0 after the onset, as in the paper)
thetas = {}   # (variant, key) -> array of 6 angles
for variant, mtype in (("h1fix", "paral"), ("h2fix", "ortho")):
    per_seed = [compute_thetas_avg(models[variant, s], mtype, n_episodes=5,
                                   random_seed=100 + s) for s in SEEDS]
    for key in ANGLE_KEYS:
        thetas[variant, key] = np.array([r[key][0] for r in per_seed])

paper_means = {"theta_self_other": (30.8, 78.9),
               "theta_self_out": (44.0, 43.2),
               "theta_other_out": (43.5, 59.1)}
print(f"{'angle':24s} {'H1 (ours, n=6)':>16s} {'H1 (paper)':>11s} "
      f"{'H2 (ours, n=6)':>16s} {'H2 (paper)':>11s}")
for key, label in ANGLE_KEYS.items():
    h1, h2 = thetas["h1fix", key], thetas["h2fix", key]
    pm = paper_means[key]
    print(f"{label:24s} {h1.mean():6.1f} ± {h1.std():4.1f}° {pm[0]:10.1f}° "
          f"{h2.mean():6.1f} ± {h2.std():4.1f}° {pm[1]:10.1f}°")

## The distributions on a circle

Each dot is one trained network, placed at its angle on the semicircle (radial position
is only for readability). Left: the angle between the actor and observer outcome
directions for the two wirings, against the two monkeys' ACC (dashed lines). Right: for
H2, the distance of each outcome direction from the evidence axis (dotted ACC lines:
actor–SE; dashed: observer–SE).

In [ ]:
def polar_dots(ax, angles, color, label, r0=0.72, dr=0.11):
    """One dot per network on the semicircle, radially staggered for readability."""
    for i, ang in enumerate(angles):
        ax.plot(np.deg2rad(ang), r0 + dr * i, "o", ms=9, color=color,
                mec="k", mew=0.6, zorder=5)
    ax.plot([], [], "o", ms=9, color=color, mec="k", mew=0.6, label=label)
    ax.vlines(np.deg2rad(np.mean(angles)), 0, 1.4, color=color, lw=2)


def style_polar(ax):
    ax.set_thetamin(0)
    ax.set_thetamax(180)
    ax.set_thetagrids([0, 45, 90, 135, 180],
                      ["0°\n(shared)", "45°", "90°\n(independent)", "135°", "180°"],
                      fontsize=8)
    ax.set_rlim(0, 1.45)
    ax.set_yticklabels([])   # radial axis carries no meaning (dots are staggered)


ACC = {"M1": dict(both=90.83, self_se=26.25, other_se=79.05, color="green"),
       "M2": dict(both=75.97, self_se=31.96, other_se=54.40, color="magenta")}

fig, (axa, axb) = plt.subplots(
    1, 2, figsize=(10.5, 4.6), subplot_kw=dict(projection="polar"))

# (a) theta(O^Act, O^Obs): H1 vs H2 vs ACC
polar_dots(axa, thetas["h1fix", "theta_self_other"], "steelblue", "H1 networks")
polar_dots(axa, thetas["h2fix", "theta_self_other"], "salmon", "H2 networks")
for m, d in ACC.items():
    axa.vlines(np.deg2rad(d["both"]), 0, 1.4, color=d["color"], ls="--", lw=1.5)
    axa.plot([], [], color=d["color"], ls="--",
             label=f"ACC {m} ({d['both']:.0f}°)")
style_polar(axa)
axa.set_title(r"$\theta(O^{Act}, O^{Obs})$: shared or separate?", fontsize=11, pad=22)
axa.legend(loc="lower left", bbox_to_anchor=(-0.35, -0.18), fontsize=8)

# (b) theta(O, SE) for H2: who can move the belief?
polar_dots(axb, thetas["h2fix", "theta_self_out"], "royalblue",
           r"H2: $\theta(O^{Act}, SE)$")
polar_dots(axb, thetas["h2fix", "theta_other_out"], "lightcoral",
           r"H2: $\theta(O^{Obs}, SE)$", r0=0.77)
for m, d in ACC.items():
    axb.vlines(np.deg2rad(d["self_se"]), 0, 1.4, color=d["color"], ls=":", lw=1.5)
    axb.vlines(np.deg2rad(d["other_se"]), 0, 1.4, color=d["color"], ls="--", lw=1.5)
axb.plot([], [], color="grey", ls=":", label="ACC: actor–SE (M1/M2)")
axb.plot([], [], color="grey", ls="--", label="ACC: observer–SE (M1/M2)")
style_polar(axb)
axb.set_title(r"$\theta(O, SE)$: who can move the belief? (H2)", fontsize=11, pad=22)
axb.legend(loc="lower left", bbox_to_anchor=(-0.35, -0.18), fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "v2_geometry.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary: the RNN results

Both wiring hypotheses produce working networks; only one looks like the brain.

| angle | H1 networks (paper) | H2 networks (paper) | ACC M1 | ACC M2 |
|---|---|---|---|---|
| $\theta(O^{Act}, O^{Obs})$ | 30.8° ± 6.9 | **78.9° ± 7.6** | 90.8° | 76.0° |
| $\theta(O^{Act}, SE)$ | 44.0° | 43.2° | 26.3° | 32.0° |
| $\theta(O^{Obs}, SE)$ | 43.5° | **59.1°** | 79.1° | 54.4° |

* **Geometry I — shared or separate?** H1's shared input forces the two outcome streams
  into nearly the *same* direction; H2's separate inputs let them fall into
  *near-orthogonal* directions, matching the ACC's 76–91°.
* **Geometry II — who moves the belief?** Under H1 both streams sit at the same distance
  from the evidence axis, so they must drive the belief equally — wrong. Under H2 the
  observer direction sits *farther* from the evidence axis than the actor direction, the
  same ordering as ACC (54–79° vs 26–32°). Observed outcomes therefore move the belief
  less, which is exactly the behavioural asymmetry.

So the ACC geometry is what you get for free when experienced and observed outcomes simply
enter the circuit through **separate input channels** — no special mechanism required.

Our own six networks per wiring (printed and plotted above) land on the same side of both
tests. **Deviations from the paper's setting:** we follow the paper's equation
($\tau = 5$ steps) and train 20k iterations instead of 100k, with six networks per wiring
instead of a hundred.